In [0]:
import importlib
import src.data_quality as dq

importlib.reload(dq)
spark.sql("USE CATALOG dbr_dev_ua5816bd")
silver = spark.table("viktoriia_kalenichenko_silver.yield_and_climate")

print("Silver rows:", silver.count())

Silver rows: 3245


In [0]:
# Completeness
completeness = dq.check_completeness(silver,["AREA", "ITEM", "YEAR", "PRODUCTION_QUANTITY"])

# Uniqueness
uniqueness = dq.check_uniqueness(silver,["AREA", "ITEM", "YEAR"])

# Validity
validity = dq.check_validity(silver)

# Display results
for result in [completeness, uniqueness, validity]:
    print(result)

{'check_name': 'completeness', 'total_rows': 3245, 'missing_values': {'AREA': 0, 'ITEM': 0, 'YEAR': 0, 'PRODUCTION_QUANTITY': 0}, 'passed': True}
{'check_name': 'uniqueness', 'total_rows': 3245, 'unique_keys': 3245, 'duplicate_rows': 0, 'passed': True}
{'check_name': 'validity', 'invalid_year': 0, 'negative_production': 0, 'negative_area': 0, 'passed': True}


In [0]:
from pyspark.sql import functions as F

# Load the Silver country reference table
countries = spark.table("viktoriia_kalenichenko_silver.dim_country_scd1")

# Apply the same business rule as in Lab 6
agriculture_filtered = silver.filter(F.trim(F.col("AREA")) != "Russian Federation")

# Check country references
consistency = dq.check_consistency(agriculture_filtered, countries, "AREA",  "AREA")

print(consistency)

{'check_name': 'consistency', 'unmatched_rows': 0, 'passed': True}


In [0]:
from pyspark.sql import functions as F

# Load the Gold fact table
gold = spark.table("viktoriia_kalenichenko_gold.fact_agri_environment")

# Compare row counts
row_reconciliation = dq.reconcile_counts(
    expected_rows=agriculture_filtered.count(),
    actual_rows=gold.count(),
)

# Compare production totals
silver_production = agriculture_filtered.agg(F.sum("PRODUCTION_QUANTITY")).first()[0]

gold_production = gold.agg(F.sum("production_quantity")).first()[0]

production_reconciliation = dq.reconcile_aggregates(
    expected_total=silver_production,
    actual_total=gold_production,
)

print(row_reconciliation)
print(production_reconciliation)

{'check_name': 'row_reconciliation', 'expected_rows': 3152, 'actual_rows': 3152, 'difference': 0, 'passed': True}
{'check_name': 'aggregate_reconciliation', 'expected_total': 10190461480.079998, 'actual_total': 10190461480.079998, 'difference': 0.0, 'tolerance': 0.01, 'passed': True}


In [0]:


# Bronze → Silver reconciliation

bronze_agriculture = spark.table("viktoriia_kalenichenko_bronze.yield_and_climate")

bronze_environment = spark.table("viktoriia_kalenichenko_bronze.environment")

silver_environment = spark.table("viktoriia_kalenichenko_silver.environment")

agriculture_reconciliation = dq.reconcile_counts(
    expected_rows=bronze_agriculture.count(),
    actual_rows=silver.count()
)

environment_reconciliation = dq.reconcile_counts(
    expected_rows=bronze_environment.count(),
    actual_rows=silver_environment.count()
)

print("Bronze → Silver agriculture:")
print(agriculture_reconciliation)

print("Bronze → Silver environment:")
print(environment_reconciliation)


Bronze → Silver agriculture:
{'check_name': 'row_reconciliation', 'expected_rows': 3245, 'actual_rows': 3245, 'difference': 0, 'passed': True}
Bronze → Silver environment:
{'check_name': 'row_reconciliation', 'expected_rows': 1196, 'actual_rows': 1196, 'difference': 0, 'passed': True}


In [0]:

# Bronze → Silver production aggregate reconciliation

bronze_production = bronze_agriculture.agg(
    F.sum("PRODUCTION_QUANTITY")
).first()[0]

silver_total_production = silver.agg(
    F.sum("PRODUCTION_QUANTITY")
).first()[0]

bronze_silver_production_reconciliation = dq.reconcile_aggregates(
    expected_total=bronze_production,
    actual_total=silver_total_production
)

print("Bronze → Silver production:")
print(bronze_silver_production_reconciliation)


Bronze → Silver production:
{'check_name': 'aggregate_reconciliation', 'expected_total': 12598438956.64, 'actual_total': 12598438956.64, 'difference': 0.0, 'tolerance': 0.01, 'passed': True}


In [0]:

# Gold dimension Data Quality checks

gold_country = spark.table("viktoriia_kalenichenko_gold.dim_country")

gold_crop = spark.table("viktoriia_kalenichenko_gold.dim_crop")

gold_year = spark.table(
    "viktoriia_kalenichenko_gold.dim_year"
)

# Check unique primary keys
country_uniqueness = dq.check_uniqueness(
    gold_country, ["country_code"]
)

crop_uniqueness = dq.check_uniqueness(
    gold_crop, ["crop_key"]
)

year_uniqueness = dq.check_uniqueness(
    gold_year, ["year_key"]
)

# Check primary keys are not NULL
country_completeness = dq.check_completeness(
    gold_country, ["country_code"]
)

crop_completeness = dq.check_completeness(
    gold_crop, ["crop_key"]
)

year_completeness = dq.check_completeness(
    gold_year, ["year_key"]
)

for name, result in [
    ("Country uniqueness", country_uniqueness),
    ("Crop uniqueness", crop_uniqueness),
    ("Year uniqueness", year_uniqueness),
    ("Country completeness", country_completeness),
    ("Crop completeness", crop_completeness),
    ("Year completeness", year_completeness),
]:
    print(name, ":", result)


Country uniqueness : {'check_name': 'uniqueness', 'total_rows': 38, 'unique_keys': 38, 'duplicate_rows': 0, 'passed': True}
Crop uniqueness : {'check_name': 'uniqueness', 'total_rows': 3, 'unique_keys': 3, 'duplicate_rows': 0, 'passed': True}
Year uniqueness : {'check_name': 'uniqueness', 'total_rows': 33, 'unique_keys': 33, 'duplicate_rows': 0, 'passed': True}
Country completeness : {'check_name': 'completeness', 'total_rows': 38, 'missing_values': {'country_code': 0}, 'passed': True}
Crop completeness : {'check_name': 'completeness', 'total_rows': 3, 'missing_values': {'crop_key': 0}, 'passed': True}
Year completeness : {'check_name': 'completeness', 'total_rows': 33, 'missing_values': {'year_key': 0}, 'passed': True}


In [0]:

# Gold fact → dimension referential integrity

# 1. Country foreign key
country_fk = dq.check_consistency(
    gold,
    gold_country,
    "country_key",
    "country_code"
)

# 2. Crop foreign key
crop_fk = dq.check_consistency(
    gold,
    gold_crop,
    "crop_key",
    "crop_key"
)

# 3. Year foreign key
year_fk = dq.check_consistency(gold,gold_year,"year_key","year_key")

print("Country FK:", country_fk)
print("Crop FK:", crop_fk)
print("Year FK:", year_fk)


Country FK: {'check_name': 'consistency', 'unmatched_rows': 0, 'passed': True}
Crop FK: {'check_name': 'consistency', 'unmatched_rows': 0, 'passed': True}
Year FK: {'check_name': 'consistency', 'unmatched_rows': 0, 'passed': True}


In [0]:

# Check completeness of Gold fact foreign keys
gold_fk_completeness = dq.check_completeness(gold,["country_key", "crop_key", "year_key"])
print("Gold FK completeness:", gold_fk_completeness)


Gold FK completeness: {'check_name': 'completeness', 'total_rows': 3152, 'missing_values': {'country_key': 0, 'crop_key': 0, 'year_key': 0}, 'passed': True}


In [0]:

from datetime import datetime, timezone

# Latest Delta table update
history = spark.sql("""
    DESCRIBE HISTORY
    dbr_dev_ua5816bd.viktoriia_kalenichenko_silver.yield_and_climate
    LIMIT 1
""")

last_update = history.select("timestamp").first()[0]

if last_update is None:
    raise RuntimeError("Delta history has no update timestamp")

# Spark returns a datetime without timezone information.
# Interpret the Delta history timestamp as UTC.
if last_update.tzinfo is None:
    last_update = last_update.replace(tzinfo=timezone.utc)

now = datetime.now(timezone.utc)
age_days = (now - last_update).total_seconds() / 86400

freshness = {
    "check_name": "freshness",
    "last_update": last_update.isoformat(),
    "age_days": round(age_days, 2),
    "max_age_days": 30,
    "passed": 0 <= age_days <= 30,
}

print(freshness)



{'check_name': 'freshness', 'last_update': '2026-10-03T09:35:47+00:00', 'age_days': 28.94, 'max_age_days': 30, 'passed': True}


In [0]:

# Collect all Data Quality check results
dq_results = [
    # Silver Data Quality
    completeness,
    uniqueness,
    validity,
    consistency,

    # Silver → Gold reconciliation
    row_reconciliation,
    production_reconciliation,

    # Bronze → Silver reconciliation
    agriculture_reconciliation,
    environment_reconciliation,
    bronze_silver_production_reconciliation,

    # Gold dimension uniqueness
    country_uniqueness,
    crop_uniqueness,
    year_uniqueness,

    # Gold dimension completeness
    country_completeness,
    crop_completeness,
    year_completeness,

    # Gold fact referential integrity
    country_fk,
    crop_fk,
    year_fk,

    # Gold fact foreign key completeness
    gold_fk_completeness,


    # Timeliness / Freshness
    freshness,

]

# Identify failed checks
failed_checks = [
    f"Check #{i}: {result['check_name']}"
    for i, result in enumerate(dq_results, start=1)
    if not result["passed"]
]

# Fail the notebook if any DQ check fails
if failed_checks:
    raise RuntimeError(
        "Data Quality checks failed:\n"
        + "\n".join(failed_checks)
    )

print(f"All {len(dq_results)} Data Quality checks PASSED!")


All 20 Data Quality checks PASSED!
